# 2025/03/18 Fairdom seek upload

This notebook updates our previous notebook by uploading to CBGP seek instead of the Fairdom instance in Seek.



In [15]:
import requests
from typing import List, Dict, Any
import pandas as pd
import time

# Define API endpoint and authentication
# CBGP
api_url = "https://seek.admin.cbgp.upm.es"
api_key = "B7qUAvsSVPX2qrlZ11azFveVt-lLhET_sRqzNiNc"
headers = {"Authorization": f"Bearer {api_key}", "Accept": "application/vnd.api+json",
           "Accept-Charset": "ISO-8859-1"}


In [6]:
def add_samples(collection:List[dict], sample_id_field: str, verbose=True):
    """
    Add samples
    ===

    Adds samples from collection. It handles the 500 errors that arise 
    in Seek 1.16, and it provides a table with the latest added records, joining
    their original identifiers with their record id in the Seek server.

    arguments:
    ---

    collection: List
        List of dictionaries containing
    sample_id_field: str
        String to idifentify the sample
    verbose: bool


    """
    out = []
    for i, sample in enumerate(collection):
        sample_id = sample['data']['attributes']['attribute_map'][sample_id_field]
        for _ in range(5):
            r = requests.post(f"{api_url}/samples", headers=headers, json=sample)

            
            if r.status_code == 200:
                break
            elif r.status_code == 500:
                # Important to avoid 500 errors.
                time.sleep(0.5)
            else:
                break


        if r.status_code == 200:
            record = r.json()
            record_id = record['data']['id']
            out.append([sample_id, record_id])
            if verbose:    
                print("sample # {0}, record {1}, added".format(
                    sample_id, 
                    record_id)
                )
        else:
            print("sample # {0}, could not be added, ERROR {1}".format(sample_id, r.status_code))
    return out


def get_all_samples_from_type(sample_type_id:int, verbose:bool=True):
    """
    Get All Samples from Type
    ===

    Seek does not provide a search api for samples. Therefore, the best way to do searches is
    to use this function to fetch all the samples belonging to certain type, and then making data
    queries locally through pandas or similar.

    arguments:
    ---

    sample_type_id: int
        Record id corresponding to the sample type

    verbose: bool


    """
    try:
        samples = requests.get(f"{api_url}/sample_types/{sample_type_id}", headers=headers).json()['data']['relationships']['samples']['data']
    except KeyError:
        raise IOError(f"Is {sample_type_id} defined??")
    
    out_samples = []
    for sample in samples:
        for _ in range(5):
            r = requests.get(f"{api_url}/samples/{sample['id']}", headers=headers)
            if r.status_code == 200:
                break
            elif r.status_code == 500:
                # Important to avoid 500 errors.
                time.sleep(0.5)
            else:
                break
        if r.status_code == 200:
            out_samples.append(r.json())
            if verbose:
                print("sample with id {0}, retrieved ".format(sample['id']))
        elif r.status_code == 500 and verbose:
            print("sample with id {0} could not be retrieved, code{1} ".format(sample['id'], s.status_code))
    return out_samples

##

## Gathering data

To create a fair isa record for our experiment, first we need to have under good control all the experimental information that we will use. The following blocks attempt to process experimental data to get it in the right format.

Tasks:

1. Read RNA library and Field Sample data.
2. Create a Field Sample id for each Field Sample by connecting the site code and the date
3. Extract "Habitat info" from RNA library and add it to Field Sample data.
4. Create a table connecting Libraries with their field samples.

### Task 1. Reading RNA libaries and Field Sample data

No issues here. Just reading XLSX files containing the info. 

**We will use Upper case for the first letter of the field, and lower case for all the others**

In [20]:
field_samples = pd.read_excel("../data/fairdom-seek-mcleish2024/FieldSampling-2025-02-20.xlsx", sheet_name="Samples")
field_samples['Season'] = field_samples['Collection Codes'].apply(lambda x: {"F":"fall", "V":"summer", "P":"spring"}[x[-1]])
field_samples.columns = ["Collection_code", "Site_code", "Date", "Town", "Longitude", "Latitude", "Season"]
field_samples

,Collection_code,Site_code,Date,Town,Longitude,Latitude,Season
0,C1F,C1,2016-02-01,Aranjuez,-3.593308,40.051302,fall
1,C3F,C1,2017-01-30,Aranjuez,-3.593308,40.051302,fall
2,C2F,C2,2016-02-01,Aranjuez,-3.599064,40.043193,fall
3,C4F,C2,2017-01-30,Aranjuez,-3.599064,40.043193,fall
4,E1F,E1,2015-11-19,Aranjuez,-3.500323,40.059138,fall
...,...,...,...,...,...,...,...
73,Q4P,Q4,2017-05-03,Mondéjar,-3.137145,40.345348,spring
74,Z1V,Z1,2015-07-16,Villaconejos,-3.476076,40.050052,summer
75,Z3V,Z1,2016-07-20,Villaconejos,-3.476076,40.050052,summer
76,Z2V,Z2,2015-07-24,Villamanrique de Tajo,-3.131000,40.044720,summer


In [19]:
library_samples = pd.read_excel("../data/fairdom-seek-mcleish2024/RNAlibrary-2025-02-20.xlsx", sheet_name="Samples")
library_samples.columns = ["Library_code", "Collection_code", "Species_name", "Habitat", "Number_of_extracts"]
library_samples

,Library_code,Collection_code,Species_name,Habitat,Number_of_extracts
0,PV001,M1V,Amaranthus sp,Crop,8
1,PV002,M1V,Convolvulus arvensis,Crop,11
2,PV003,M1V,Cucumis melo,Crop,13
3,PV003bgi,M1V,Cucumis melo,Crop,13
4,PV004bgi,M1V,Cyperus longus,Crop,7
...,...,...,...,...,...
318,PV586,H1P,Fumaria parviflora,Crop,8
319,PV587,H1P,Hirschfeldia incana,Crop,8
320,PV588,H1P,Hordeum vulgare,Crop,8
321,PV589,H1P,Hordeum vulgare,Crop,8


### Task 2. Extracting habitat info

In [26]:
habitat_type = library_samples[['Collection_code', 'Habitat']].drop_duplicates(keep='first').sort_values(by='Collection_code')
habitat_type = habitat_type.reset_index()[['Collection_code', 'Habitat']]
field_samples = pd.merge(field_samples, habitat_type, how='left', left_on=['Collection_code'], right_on=['Collection_code'])
field_samples['Habitat'] = field_samples['Habitat'].fillna("null")
field_samples

,Collection_code,Site_code,Date,Town,Longitude,Latitude,Season,Sample code,Habitat
0,C1F,C1,2016-02-01,Aranjuez,-3.593308,40.051302,fall,C1-01/02/2016,Crop
1,C3F,C1,2017-01-30,Aranjuez,-3.593308,40.051302,fall,C1-30/01/2017,null
2,C2F,C2,2016-02-01,Aranjuez,-3.599064,40.043193,fall,C2-01/02/2016,Crop
3,C4F,C2,2017-01-30,Aranjuez,-3.599064,40.043193,fall,C2-30/01/2017,null
4,E1F,E1,2015-11-19,Aranjuez,-3.500323,40.059138,fall,E1-19/11/2015,Wasteland
...,...,...,...,...,...,...,...,...,...
73,Q4P,Q4,2017-05-03,Mondéjar,-3.137145,40.345348,spring,Q4-03/05/2017,Oak
74,Z1V,Z1,2015-07-16,Villaconejos,-3.476076,40.050052,summer,Z1-16/07/2015,Crop
75,Z3V,Z1,2016-07-20,Villaconejos,-3.476076,40.050052,summer,Z1-20/07/2016,null
76,Z2V,Z2,2015-07-24,Villamanrique de Tajo,-3.131000,40.044720,summer,Z2-24/07/2015,Crop


### Task 3. Creating an index for field samples

In [28]:
field_samples['Sample code'] = field_samples.apply(lambda x: x['Site_code'] + '-' + x['Date'].strftime('%d/%m/%Y'), axis=1)
field_samples[['Sample code', 'Site_code', 'Town', 'Longitude', 'Latitude', 'Habitat', 'Date', 'Season']]

,Sample code,Site_code,Town,Longitude,Latitude,Habitat,Date,Season
0,C1-01/02/2016,C1,Aranjuez,-3.593308,40.051302,Crop,2016-02-01,fall
1,C1-30/01/2017,C1,Aranjuez,-3.593308,40.051302,null,2017-01-30,fall
2,C2-01/02/2016,C2,Aranjuez,-3.599064,40.043193,Crop,2016-02-01,fall
3,C2-30/01/2017,C2,Aranjuez,-3.599064,40.043193,null,2017-01-30,fall
4,E1-19/11/2015,E1,Aranjuez,-3.500323,40.059138,Wasteland,2015-11-19,fall
...,...,...,...,...,...,...,...,...
73,Q4-03/05/2017,Q4,Mondéjar,-3.137145,40.345348,Oak,2017-05-03,spring
74,Z1-16/07/2015,Z1,Villaconejos,-3.476076,40.050052,Crop,2015-07-16,summer
75,Z1-20/07/2016,Z1,Villaconejos,-3.476076,40.050052,null,2016-07-20,summer
76,Z2-24/07/2015,Z2,Villamanrique de Tajo,-3.131000,40.044720,Crop,2015-07-24,summer


### Task 4. Connecting Field samples and habitats

In [ ]:
collection_code_column = pd.merge(
    library_samples[['Collection_code', 'Library_code']], 
    field_samples[['Collection_code', 'Sample_code']], on=['Collection_code']
).groupby('Library_code', as_index=True)['Sample_code'].apply(list).reset_index()# .rename(columns={'Sample code':'Field samples'})
library_samples = pd.merge(
    library_samples, collection_code_column, on=['Library_code']
)
library_samples

,Library_code,Collection_code,Species_name,Habitat,Number_of_extracts,Sample_code
0,PV001,M1V,Amaranthus sp,Crop,8,[M1-16/07/2015]
1,PV002,M1V,Convolvulus arvensis,Crop,11,[M1-16/07/2015]
2,PV003,M1V,Cucumis melo,Crop,13,[M1-16/07/2015]
3,PV003bgi,M1V,Cucumis melo,Crop,13,[M1-16/07/2015]
4,PV004bgi,M1V,Cyperus longus,Crop,7,[M1-16/07/2015]
...,...,...,...,...,...,...
308,PV586,H1P,Fumaria parviflora,Crop,8,[H1-21/04/2016]
309,PV587,H1P,Hirschfeldia incana,Crop,8,[H1-21/04/2016]
310,PV588,H1P,Hordeum vulgare,Crop,8,[H1-21/04/2016]
311,PV589,H1P,Hordeum vulgare,Crop,8,[H1-21/04/2016]


In [13]:
library_samples.query('Collection_code == "E1P"')

,Library_code,Collection_code,Species_name,Habitat,Number_of_extracts,Sample_code
235,PV277,E1P,Anacyclus clavatus,Wasteland,5,"[E1-23/05/2016, E1-03/05/2017]"
236,PV296,E1P,Stipa parviflora,Wasteland,6,"[E1-23/05/2016, E1-03/05/2017]"
237,PV298,E1P,Teucrium capitatum,Wasteland,15,"[E1-23/05/2016, E1-03/05/2017]"
297,PV572,E1P,Trifolium tomentosum,Wasteland,3,"[E1-23/05/2016, E1-03/05/2017]"
305,PV583,E1P,Astragalus sesameus,Wasteland,6,"[E1-23/05/2016, E1-03/05/2017]"


## Uploading samples

We will proceed iteratively to upload forward-linked samples. Ideally, we would like this process to take place in a single run. However, this is not usually the case, as the Seek interface is pretty slow. 

Therefore, we will always proceed as: post instances, fetch identifiers, use those identifiers in the next post.

Tasks:

1. Posting field sample records.
2. Posting library records.
3. Posting extracts.
4. Posting raw reads


### Useful code

We will create each of the sample records by composing a function that it's specific of each sample type, and a function that works with all sample types.

In [29]:
def create_record(attributes_dict: Dict[str, Any], project_id, people_id, sample_type_id, assay_id):
    data = {
        'data':{
            "type": "samples",
            "attributes": attributes_dict,
            "relationships": {
                "sample_type": {"data": {"id": sample_type_id,"type": "sample_types"}},
                "creators": {"data": [{"id": people_id, "type": "people"}]},
                "projects": {"data": [{"id": project_id, "type": "projects"}]},
                "assays": {"data": [{"id": assay_id, "type": "assays"}]}
            }
        }   
    }
    return data

### Task 1. Posting "Field sample" records

In [58]:
def create_field_attributes(entry: Dict[str, Any], project_id: str, people_id: str, sample_type_id: str, assay_id: str):
    attributes = {
        
        "attribute_map": {
            "Sample code": entry['Sample_code'],
            "Location code": entry['Site_code'],
            "Town": entry['Town'],
            "Longitude": entry['Longitude'],
            "Latitude": entry['Latitude'],
            "Habitat": entry['Habitat'],
            "Date": entry['Date'].strftime('%d/%m/%Y'),
            "Season": entry['Season']
        },
        "tags": [
            entry['Habitat'],
            entry['Season'], 
            "field sample"
        ]
    }    
    
    record = create_record(
        attributes_dict=attributes, project_id=project_id, people_id=people_id,
        sample_type_id=sample_type_id, assay_id=assay_id
    )
    return record

In [59]:
# IMPORTANT TO CHECK THESE ONES BEFORE UPLOAD
people_id = '1'
project_id = '2'
assay_id = '1' # Check these ones carefully
sample_type_id = '1'

create_field_attributes_ = lambda x: create_field_attributes(
    x, 
    project_id=project_id, people_id=people_id, 
    sample_type_id=sample_type_id, assay_id=assay_id
)
field_sample_records = list(map(
    create_field_attributes_, field_samples.to_dict(orient='records'))
)
field_sample_records[0]

{'data': {'type': 'samples',
  'attributes': {'attribute_map': {'Sample code': 'C1-01/02/2016',
    'Location code': 'C1',
    'Town': 'Aranjuez',
    'Longitude': -3.593308,
    'Latitude': 40.051302,
    'Habitat': 'Crop',
    'Date': '01/02/2016',
    'Season': 'fall'},
   'tags': ['Crop', 'fall', 'field sample']},
  'relationships': {'sample_type': {'data': {'id': '1',
     'type': 'sample_types'}},
   'creators': {'data': [{'id': '1', 'type': 'people'}]},
   'projects': {'data': [{'id': '2', 'type': 'projects'}]},
   'assays': {'data': [{'id': '1', 'type': 'assays'}]}}}}

In [62]:
field_sample_records = add_samples(field_sample_records, "Sample code")

sample # C1-01/02/2016, record 1, added
sample # C1-30/01/2017, record 2, added
sample # C2-01/02/2016, record 3, added
sample # C2-30/01/2017, record 4, added
sample # E1-19/11/2015, record 5, added
sample # E1-10/11/2016, record 6, added
sample # E1-23/05/2016, record 7, added
sample # E1-03/05/2017, record 8, added
sample # E2-19/11/2015, record 9, added
sample # E2-10/11/2016, record 10, added
sample # E2-23/05/2016, record 11, added
sample # E2-10/05/2017, record 12, added
sample # E3-01/02/2016, record 13, added
sample # E3-10/11/2016, record 14, added
sample # E3-31/05/2016, record 15, added
sample # E3-03/05/2017, record 16, added
sample # E4-04/11/2015, record 17, added
sample # E4-27/10/2016, record 18, added
sample # E4-31/05/2016, record 19, added
sample # E4-03/05/2017, record 20, added
sample # H1-21/04/2016, record 21, added
sample # H1-19/04/2017, record 22, added
sample # H2-21/04/2016, record 23, added
sample # H2-19/04/2017, record 24, added
sample # H3-21/04/2016, r

In [30]:
field_sample_records = get_all_samples_from_type('1', verbose=False)
field_sample_records = [dict(record_id=item['data']['id'], sample_id=item['data']['attributes']['attribute_map']['Sample code']) for item in field_sample_records]


In [31]:
field_sample_records = pd.DataFrame.from_records(field_sample_records)
field_sample_records

,record_id,sample_id
0,507,C1-01/02/2016
1,508,C1-30/01/2017
2,509,C2-01/02/2016
3,510,C2-30/01/2017
4,511,E1-19/11/2015
...,...,...
73,580,Q4-03/05/2017
74,581,Z1-16/07/2015
75,582,Z1-20/07/2016
76,583,Z2-24/07/2015


## Task 2. Create RNA extracts

In [32]:
field_sample_records['Extract code'] = field_sample_records['sample_id'].apply(lambda x: f'RNA-{x}')
field_sample_records


,record_id,sample_id,Extract code
0,507,C1-01/02/2016,RNA-C1-01/02/2016
1,508,C1-30/01/2017,RNA-C1-30/01/2017
2,509,C2-01/02/2016,RNA-C2-01/02/2016
3,510,C2-30/01/2017,RNA-C2-30/01/2017
4,511,E1-19/11/2015,RNA-E1-19/11/2015
...,...,...,...
73,580,Q4-03/05/2017,RNA-Q4-03/05/2017
74,581,Z1-16/07/2015,RNA-Z1-16/07/2015
75,582,Z1-20/07/2016,RNA-Z1-20/07/2016
76,583,Z2-24/07/2015,RNA-Z2-24/07/2015


In [33]:
def create_rna_extract_record(entry: Dict[str, Any], project_id: str, people_id: str, sample_type_id: str, assay_id: str):
    attributes = {
        
        "attribute_map": {
            "RNA extract code": entry["Extract code"],
            "Field sample": entry["record_id"],
        },
        "tags": []
    }    
    
    record = create_record(
        attributes_dict=attributes, project_id=project_id, people_id=people_id,
        sample_type_id=sample_type_id, assay_id=assay_id
    )
    return record


# IMPORTANT TO CHECK THESE ONES BEFORE UPLOAD
people_id = '1'
project_id = '2'
assay_id = '2' # Check these ones carefully
sample_type_id = '5'

create_rna_extract_record_ = lambda x: create_rna_extract_record(
    x,people_id=people_id, project_id=project_id, assay_id=assay_id, sample_type_id=sample_type_id
)


In [34]:
extract_records = list(map(create_rna_extract_record_, field_sample_records.to_dict(orient='records')))
extract_records[24]


{'data': {'type': 'samples',
  'attributes': {'attribute_map': {'RNA extract code': 'RNA-H3-21/04/2016',
    'Field sample': '531'},
   'tags': []},
  'relationships': {'sample_type': {'data': {'id': '5',
     'type': 'sample_types'}},
   'creators': {'data': [{'id': '1', 'type': 'people'}]},
   'projects': {'data': [{'id': '2', 'type': 'projects'}]},
   'assays': {'data': [{'id': '2', 'type': 'assays'}]}}}}

In [ ]:
extract_records = add_samples(extract_records, "RNA extract code")

sample # RNA-C1-01/02/2016, record 585, added
sample # RNA-C1-30/01/2017, record 586, added
sample # RNA-C2-01/02/2016, record 587, added
sample # RNA-C2-30/01/2017, record 588, added
sample # RNA-E1-19/11/2015, record 589, added
sample # RNA-E1-10/11/2016, record 590, added
sample # RNA-E1-23/05/2016, record 591, added
sample # RNA-E1-03/05/2017, record 592, added
sample # RNA-E2-19/11/2015, record 593, added
sample # RNA-E2-10/11/2016, record 594, added
sample # RNA-E2-23/05/2016, record 595, added
sample # RNA-E2-10/05/2017, record 596, added
sample # RNA-E3-01/02/2016, record 597, added
sample # RNA-E3-10/11/2016, record 598, added
sample # RNA-E3-31/05/2016, record 599, added
sample # RNA-E3-03/05/2017, record 600, added
sample # RNA-E4-04/11/2015, record 601, added
sample # RNA-E4-27/10/2016, record 602, added
sample # RNA-E4-31/05/2016, record 603, added
sample # RNA-E4-03/05/2017, record 604, added
sample # RNA-H1-21/04/2016, record 605, added
sample # RNA-H1-19/04/2017, record

### Task 2. Creating libraries


This task has two sub-tasks: 
1. Converting sample_ids in "Library_samples" to record id
2. Posting

#### Converting Sample ids to Record ids

In [73]:
field_sample_records_bysample_id = field_sample_records.set_index('sample_id')['record_id'].to_dict()
field_sample_records_bysample_id['M1-16/07/2015']

'49'

In [72]:
library_samples

,Library_code,Collection_code,Species_name,Habitat,Number_of_extracts,Sample_code
0,PV001,M1V,Amaranthus sp,Crop,8,[M1-16/07/2015]
1,PV002,M1V,Convolvulus arvensis,Crop,11,[M1-16/07/2015]
2,PV003,M1V,Cucumis melo,Crop,13,[M1-16/07/2015]
3,PV003bgi,M1V,Cucumis melo,Crop,13,[M1-16/07/2015]
4,PV004bgi,M1V,Cyperus longus,Crop,7,[M1-16/07/2015]
...,...,...,...,...,...,...
308,PV586,H1P,Fumaria parviflora,Crop,8,[H1-21/04/2016]
309,PV587,H1P,Hirschfeldia incana,Crop,8,[H1-21/04/2016]
310,PV588,H1P,Hordeum vulgare,Crop,8,[H1-21/04/2016]
311,PV589,H1P,Hordeum vulgare,Crop,8,[H1-21/04/2016]


In [75]:
library_samples['Sample_record'] = library_samples['Sample_code'].apply(lambda x: list(map(lambda x: str(field_sample_records_bysample_id[x]), x)))
library_samples[:10]

,Library_code,Collection_code,Species_name,Habitat,Number_of_extracts,Sample_code,Sample_record
0,PV001,M1V,Amaranthus sp,Crop,8,[M1-16/07/2015],[49]
1,PV002,M1V,Convolvulus arvensis,Crop,11,[M1-16/07/2015],[49]
2,PV003,M1V,Cucumis melo,Crop,13,[M1-16/07/2015],[49]
3,PV003bgi,M1V,Cucumis melo,Crop,13,[M1-16/07/2015],[49]
4,PV004bgi,M1V,Cyperus longus,Crop,7,[M1-16/07/2015],[49]
5,PV005bgi,Q1F,Artemisia herba alba,Oak,12,"[Q1-08/09/2015, Q1-12/09/2016]","[59, 60]"
6,PV006bgi,Q1F,Artemisia herba alba,Oak,19,"[Q1-08/09/2015, Q1-12/09/2016]","[59, 60]"
7,PV007bgi,Q1F,Teucrium pseudochamaepitys,Oak,17,"[Q1-08/09/2015, Q1-12/09/2016]","[59, 60]"
8,PV008bgi,Q1F,Jasminum fruticans,Oak,11,"[Q1-08/09/2015, Q1-12/09/2016]","[59, 60]"
9,PV009,Q1F,Quercus coccifera,Oak,13,"[Q1-08/09/2015, Q1-12/09/2016]","[59, 60]"


In [ ]:
def create_library_record(entry: Dict[str, Any], project_id: str, people_id: str, sample_type_id: str, assay_id: str):
    attributes = {
        
        "attribute_map": {
            "Library code": entry['Library_code'],
            "Number of extracts": entry['Number_of_extracts'],
            "Species name": entry['Species_name'],
            "RNA extract": entry['Sample_record']
        },
        "tags": [
            "library"
        ]
    }    
    
    record = create_record(
        attributes_dict=attributes, project_id=project_id, people_id=people_id,
        sample_type_id=sample_type_id, assay_id=assay_id
    )
    return record


In [86]:
# IMPORTANT TO CHECK THESE ONES BEFORE UPLOAD
people_id = '1'
project_id = '2'
assay_id = '2' # Check these ones carefully
sample_type_id = '2'

create_library_record_ = lambda x: create_library_record(
    x, 
    project_id=project_id, people_id=people_id, 
    sample_type_id=sample_type_id, assay_id=assay_id
)
library_records = list(map(
    create_library_record_, library_samples.to_dict(orient='records'))
)
library_records[24]

{'data': {'type': 'samples',
  'attributes': {'attribute_map': {'Library code': 'PV024',
    'Number of extracts': 15,
    'Species name': 'Verbascum sinuatum',
    'Field samples': ['17', '18']},
   'tags': ['library']},
  'relationships': {'sample_type': {'data': {'id': '2',
     'type': 'sample_types'}},
   'creators': {'data': [{'id': '1', 'type': 'people'}]},
   'projects': {'data': [{'id': '2', 'type': 'projects'}]},
   'assays': {'data': [{'id': '2', 'type': 'assays'}]}}}}

In [87]:
len(library_records)

313

In [88]:
library_records = add_samples(library_records, "Library code")

sample # PV001, record 79, added
sample # PV002, record 80, added
sample # PV003, record 81, added
sample # PV003bgi, record 82, added
sample # PV004bgi, record 83, added
sample # PV005bgi, record 84, added
sample # PV006bgi, record 85, added
sample # PV007bgi, record 86, added
sample # PV008bgi, record 87, added
sample # PV009, record 88, added
sample # PV009bgi, record 89, added
sample # PV010bgi, record 90, added
sample # PV011, record 91, added
sample # PV012bgi, record 92, added
sample # PV013bgi, record 93, added
sample # PV014, record 94, added
sample # PV015bgi, record 95, added
sample # PV016, record 96, added
sample # PV017bgi, record 97, added
sample # PV018bgi, record 98, added
sample # PV020bgi, record 99, added
sample # PV021bgi, record 100, added
sample # PV022, record 101, added
sample # PV023, record 102, added
sample # PV024, record 103, added
sample # PV025, record 104, added
sample # PV026, record 105, added
sample # PV027, record 106, added
sample # PV028, record 1

In [89]:
library_records = get_all_samples_from_type('2', verbose=False)
library_records = [dict(record_id=item['data']['id'], sample_id=item['data']['attributes']['attribute_map']['Library code']) for item in library_records]
library_records = pd.DataFrame.from_records(library_records)
library_records


,record_id,sample_id
0,79,PV001
1,80,PV002
2,81,PV003
3,82,PV003bgi
4,83,PV004bgi
...,...,...
307,386,PV586
308,387,PV587
309,388,PV588
310,389,PV589


### Task 3. Creating RNA extracts

,record_id,sample_id,Extract code
0,79,PV001,RNA-PV001
1,80,PV002,RNA-PV002
2,81,PV003,RNA-PV003
3,82,PV003bgi,RNA-PV003bgi
4,83,PV004bgi,RNA-PV004bgi
...,...,...,...
307,386,PV586,RNA-PV586
308,387,PV587,RNA-PV587
309,388,PV588,RNA-PV588
310,389,PV589,RNA-PV589


{'data': {'type': 'samples',
  'attributes': {'attribute_map': {'RNA extract code': 'RNA-PV024',
    'Library': '103'},
   'tags': ['RNA extract']},
  'relationships': {'sample_type': {'data': {'id': '3',
     'type': 'sample_types'}},
   'creators': {'data': [{'id': '1', 'type': 'people'}]},
   'projects': {'data': [{'id': '2', 'type': 'projects'}]},
   'assays': {'data': [{'id': '3', 'type': 'assays'}]}}}}

sample # RNA-PV001, record 391, added
sample # RNA-PV002, record 392, added
sample # RNA-PV003, record 393, added
sample # RNA-PV003bgi, record 394, added
sample # RNA-PV004bgi, record 395, added
sample # RNA-PV005bgi, record 396, added
sample # RNA-PV006bgi, record 397, added
sample # RNA-PV007bgi, record 398, added
sample # RNA-PV008bgi, record 399, added
sample # RNA-PV009, record 400, added
sample # RNA-PV009bgi, record 401, added
sample # RNA-PV010bgi, record 402, added
sample # RNA-PV011, record 403, added
sample # RNA-PV012bgi, record 404, added
sample # RNA-PV013bgi, record 405, added
sample # RNA-PV014, record 406, added
sample # RNA-PV015bgi, record 407, added
sample # RNA-PV016, record 408, added
sample # RNA-PV017bgi, record 409, added
sample # RNA-PV018bgi, record 410, added
sample # RNA-PV020bgi, record 411, added
sample # RNA-PV021bgi, record 412, added
sample # RNA-PV022, record 413, added
sample # RNA-PV023, record 414, added
sample # RNA-PV024, record 415, added
sampl

### Task 4. Creating Raw reads

Two step process: uploading the files, and then creating the records.

In [116]:
files

,files,Library code,strand
0,PV051_22518_ACTTGA_read1.sub.fastq,PV051,read1
1,PV051_22518_ACTTGA_read2.sub.fastq,PV051,read2
2,PV052_22519_AGTCAA_read1.sub.fastq,PV052,read1
3,PV052_22519_AGTCAA_read2.sub.fastq,PV052,read2
4,PV053_22520_AGTTCC_read1.sub.fastq,PV053,read1
...,...,...,...
571,PV97_22531_GTTTCG_read2.sub.fastq,PV97,read2
572,PV98_22334_CGTACG_read1.sub.fastq,PV98,read1
573,PV98_22334_CGTACG_read2.sub.fastq,PV98,read2
574,PV99_22335_GAGTGG_read1.sub.fastq,PV99,read1


In [127]:
files = pd.read_csv("../samples/raw-seqs-index.csv", header=None, names=['files'])
files['Library code'] = files['files'].apply(lambda x: x.split('_')[0])
files['Primers'] = files['files'].apply(lambda x: x.split('_')[2])
files['strand'] = files['files'].apply(lambda x: x.split('_')[-1].replace('.sub.fastq', ''))
files_pvt = files.pivot(index=['Library code', 'Primers'], columns='strand', values='files')
files_pvt = files_pvt.reset_index()
files_pvt

strand,Library code,Primers,read1,read2
0,PV051,ACTTGA,PV051_22518_ACTTGA_read1.sub.fastq,PV051_22518_ACTTGA_read2.sub.fastq
1,PV052,AGTCAA,PV052_22519_AGTCAA_read1.sub.fastq,PV052_22519_AGTCAA_read2.sub.fastq
2,PV053,AGTTCC,PV053_22520_AGTTCC_read1.sub.fastq,PV053_22520_AGTTCC_read2.sub.fastq
3,PV054,ATGTCA,PV054_22521_ATGTCA_read1.sub.fastq,PV054_22521_ATGTCA_read2.sub.fastq
4,PV056,GTGGCC,PV056_22523_GTGGCC_read1.sub.fastq,PV056_22523_GTGGCC_read2.sub.fastq
...,...,...,...,...
283,PV95,GTGAAA,PV95_22331_GTGAAA_read1.sub.fastq,PV95_22331_GTGAAA_read2.sub.fastq
284,PV96,GTGGCC,PV96_22332_GTGGCC_read1.sub.fastq,PV96_22332_GTGGCC_read2.sub.fastq
285,PV97,GTTTCG,PV97_22531_GTTTCG_read1.sub.fastq,PV97_22531_GTTTCG_read2.sub.fastq
286,PV98,CGTACG,PV98_22334_CGTACG_read1.sub.fastq,PV98_22334_CGTACG_read2.sub.fastq


In [128]:
files

,files,Library code,Primers,strand
0,PV051_22518_ACTTGA_read1.sub.fastq,PV051,ACTTGA,read1
1,PV051_22518_ACTTGA_read2.sub.fastq,PV051,ACTTGA,read2
2,PV052_22519_AGTCAA_read1.sub.fastq,PV052,AGTCAA,read1
3,PV052_22519_AGTCAA_read2.sub.fastq,PV052,AGTCAA,read2
4,PV053_22520_AGTTCC_read1.sub.fastq,PV053,AGTTCC,read1
...,...,...,...,...
571,PV97_22531_GTTTCG_read2.sub.fastq,PV97,GTTTCG,read2
572,PV98_22334_CGTACG_read1.sub.fastq,PV98,CGTACG,read1
573,PV98_22334_CGTACG_read2.sub.fastq,PV98,CGTACG,read2
574,PV99_22335_GAGTGG_read1.sub.fastq,PV99,GAGTGG,read1


In [133]:
def upload_data_file(filename, path, title, assay_id, study_id, investigation_id, people_id, project_id, data_type, data_type_uri, data_format, data_format_uri):
    local_blob = {
        'original_filename': filename, 'content_type':'text/plain'
    }
    data_file = {}
    data_file['data'] = {}
    data_file['data']['type'] = 'data_files'
    data_file['data']['attributes'] = {}
    data_file['data']['attributes']['title'] = title
    data_file['data']['attributes']['license'] = 'notspecified'
    data_file['data']['attributes']['policy'] = {
        'access': 'no_access',
        'permissions': [{'resource': {'id': project_id, 'type': 'projects'},
        'access': 'manage'}]
    }
    data_file['data']['attributes']['content_blobs'] = [local_blob]
    data_file['data']['attributes']['data_type_annotations'] = [
        {'label': data_type, 'identifier': data_type_uri}
    ]
    data_file['data']['attributes']['data_format_annotations']= [
        {'label': data_format, 'identifier': data_format_uri}
    ]
    data_file['data']['relationships'] = {}
    data_file['data']['relationships']['projects'] = {'data': [{'id': project_id, 'type': 'projects'}]}
    data_file['data']['relationships']['people'] = {'data': [{'id': people_id, 'type': 'people'}]}
    data_file['data']['relationships']['investigations'] = {'data': [{'id': investigation_id, 'type': 'investigations'}]}
    data_file['data']['relationships']['studies'] = {'data': [{'id': study_id, 'type': 'studies'}]}
    data_file['data']['relationships']['assays'] = {'data': [{'id': assay_id, 'type': 'assays'}]}

    # Registering data file
    for i in range(5):
        try:
            r = requests.post(f'{api_url}/data_files/', json=data_file, headers=headers)
            r.raise_for_status()
            break
        except requests.HTTPError:
            continue

    if r.status_code == 500:
        print(f"unable to upload {filename}")

    # Obtain BLOB URL
    blob_url = r.json()['data']['attributes']['content_blobs'][0]['link']
    blob_url = blob_url.replace('http://localhost:3000', api_url)
    # Upload file to Blob
    for i in range(5):
        
        try:
            with open(path + '/' + filename, 'r') as f:
                f.seek(0)
                upload = requests.put(blob_url, files={'file':f}, headers=headers)
                upload.raise_for_status()
            break
        except requests.HTTPError:
            continue

    if r.status_code == 500:
        print(f"unable to upload {filename}")

    
    return upload
        


In [134]:
for _, row in files.iterrows():
    print("processing ", row['files'])
    upload_data_file(
        filename=row['files'], path="../samples/all_libraries/",
        title=row['files'], assay_id='4', study_id='1',
        investigation_id='1', people_id='1',
        project_id='2', data_type='Sequence',
        data_type_uri='http://edamontology.org/data_2044',
        data_format='FASTQ-illumina',
        data_format_uri='http://edamontology.org/format_1931'
    )

processing  PV051_22518_ACTTGA_read1.sub.fastq
processing  PV051_22518_ACTTGA_read2.sub.fastq
processing  PV052_22519_AGTCAA_read1.sub.fastq
processing  PV052_22519_AGTCAA_read2.sub.fastq
processing  PV053_22520_AGTTCC_read1.sub.fastq
processing  PV053_22520_AGTTCC_read2.sub.fastq
processing  PV054_22521_ATGTCA_read1.sub.fastq
processing  PV054_22521_ATGTCA_read2.sub.fastq
processing  PV056_22523_GTGGCC_read1.sub.fastq
processing  PV056_22523_GTGGCC_read2.sub.fastq
processing  PV057_22524_GTTTCG_read1.sub.fastq
processing  PV057_22524_GTTTCG_read2.sub.fastq
processing  PV058_22525_GAGTGG_read1.sub.fastq
processing  PV058_22525_GAGTGG_read2.sub.fastq
processing  PV058_22526_ATCACG_read1.sub.fastq
processing  PV058_22526_ATCACG_read2.sub.fastq
processing  PV062_22529_GCCAAT_read1.sub.fastq
processing  PV062_22529_GCCAAT_read2.sub.fastq
processing  PV063_22530_GTCCGC_read1.sub.fastq
processing  PV063_22530_GTCCGC_read2.sub.fastq
processing  PV065_21352_TAGCTT_read1.sub.fastq
processing  P

In [156]:
extract_records = get_all_samples_from_type('3', verbose=False)
extract_records = [dict(record_id=item['data']['id'], sample_id=item['data']['attributes']['attribute_map']['RNA extract code']) for item in extract_records]
extract_records = pd.DataFrame.from_records(extract_records)
extract_records


,record_id,sample_id
0,391,RNA-PV001
1,392,RNA-PV002
2,393,RNA-PV003
3,394,RNA-PV003bgi
4,395,RNA-PV004bgi
...,...,...
307,698,RNA-PV586
308,699,RNA-PV587
309,700,RNA-PV588
310,701,RNA-PV589


In [161]:
print('\n'.join(extract_records.sample_id.to_list()))

RNA-PV001
RNA-PV002
RNA-PV003
RNA-PV003bgi
RNA-PV004bgi
RNA-PV005bgi
RNA-PV006bgi
RNA-PV007bgi
RNA-PV008bgi
RNA-PV009
RNA-PV009bgi
RNA-PV010bgi
RNA-PV011
RNA-PV012bgi
RNA-PV013bgi
RNA-PV014
RNA-PV015bgi
RNA-PV016
RNA-PV017bgi
RNA-PV018bgi
RNA-PV020bgi
RNA-PV021bgi
RNA-PV022
RNA-PV023
RNA-PV024
RNA-PV025
RNA-PV026
RNA-PV027
RNA-PV028
RNA-PV029
RNA-PV030
RNA-PV031
RNA-PV032
RNA-PV033
RNA-PV034
RNA-PV035
RNA-PV036
RNA-PV037
RNA-PV041
RNA-PV042
RNA-PV046
RNA-PV047
RNA-PV048
RNA-PV049
RNA-PV050
RNA-PV051
RNA-PV052
RNA-PV053
RNA-PV054
RNA-PV055
RNA-PV056
RNA-PV057
RNA-PV058
RNA-PV059
RNA-PV060
RNA-PV061
RNA-PV062
RNA-PV063
RNA-PV064
RNA-PV065
RNA-PV066
RNA-PV068
RNA-PV069
RNA-PV070
RNA-PV071
RNA-PV072
RNA-PV074
RNA-PV075
RNA-PV076
RNA-PV078
RNA-PV079
RNA-PV080
RNA-PV081
RNA-PV082
RNA-PV083
RNA-PV084
RNA-PV085
RNA-PV086
RNA-PV087
RNA-PV088
RNA-PV089
RNA-PV090
RNA-PV091
RNA-PV092
RNA-PV093
RNA-PV094
RNA-PV095
RNA-PV096
RNA-PV097
RNA-PV098
RNA-PV099
RNA-PV100
RNA-PV101
RNA-PV102
RNA-PV103
RNA-P

In [140]:
def get_all_files(verbose:bool=True):
    
    try:
        files = requests.get(f"{api_url}/data_files/", headers=headers).json()['data']
    except KeyError:
        raise IOError(f"Is {sample_type_id} defined??")
    out = []    
    for file in files:
        out.append(
            dict(
                record_id=file['id'],
                title=file['attributes']['title']
            )
        )
    
    return out

all_file_records = get_all_files()
all_file_records = pd.DataFrame.from_records(all_file_records)
all_file_records

,record_id,title
0,582,PV99_22335_GAGTGG_read2.sub.fastq
1,581,PV99_22335_GAGTGG_read1.sub.fastq
2,580,PV98_22334_CGTACG_read2.sub.fastq
3,579,PV98_22334_CGTACG_read1.sub.fastq
4,578,PV97_22531_GTTTCG_read2.sub.fastq
...,...,...
571,11,PV053_22520_AGTTCC_read1.sub.fastq
572,10,PV052_22519_AGTCAA_read2.sub.fastq
573,9,PV052_22519_AGTCAA_read1.sub.fastq
574,8,PV051_22518_ACTTGA_read2.sub.fastq


In [142]:
all_file_records_index = all_file_records.set_index('title')['record_id'].to_dict()

In [144]:
files_pvt

strand,Library code,Primers,read1,read2
0,PV051,ACTTGA,PV051_22518_ACTTGA_read1.sub.fastq,PV051_22518_ACTTGA_read2.sub.fastq
1,PV052,AGTCAA,PV052_22519_AGTCAA_read1.sub.fastq,PV052_22519_AGTCAA_read2.sub.fastq
2,PV053,AGTTCC,PV053_22520_AGTTCC_read1.sub.fastq,PV053_22520_AGTTCC_read2.sub.fastq
3,PV054,ATGTCA,PV054_22521_ATGTCA_read1.sub.fastq,PV054_22521_ATGTCA_read2.sub.fastq
4,PV056,GTGGCC,PV056_22523_GTGGCC_read1.sub.fastq,PV056_22523_GTGGCC_read2.sub.fastq
...,...,...,...,...
283,PV95,GTGAAA,PV95_22331_GTGAAA_read1.sub.fastq,PV95_22331_GTGAAA_read2.sub.fastq
284,PV96,GTGGCC,PV96_22332_GTGGCC_read1.sub.fastq,PV96_22332_GTGGCC_read2.sub.fastq
285,PV97,GTTTCG,PV97_22531_GTTTCG_read1.sub.fastq,PV97_22531_GTTTCG_read2.sub.fastq
286,PV98,CGTACG,PV98_22334_CGTACG_read1.sub.fastq,PV98_22334_CGTACG_read2.sub.fastq


In [157]:
# extract_records = pd.DataFrame(extract_records, columns=['sample_id', 'Extract record id'])
extract_records['Library code'] = extract_records['sample_id'].apply(lambda x: x.replace('RNA-', ''))
extract_records

,record_id,sample_id,Library code
0,391,RNA-PV001,PV001
1,392,RNA-PV002,PV002
2,393,RNA-PV003,PV003
3,394,RNA-PV003bgi,PV003bgi
4,395,RNA-PV004bgi,PV004bgi
...,...,...,...
307,698,RNA-PV586,PV586
308,699,RNA-PV587,PV587
309,700,RNA-PV588,PV588
310,701,RNA-PV589,PV589


In [167]:
files_pvt['read1_id'] = files_pvt['read1'].apply(lambda x: all_file_records_index[x])
files_pvt['read2_id'] = files_pvt['read2'].apply(lambda x: all_file_records_index[x])
files_pvt['Library code'] = files_pvt['Library code'].apply(lambda x: 'PV{:03d}'.format(int(x.replace('PV', ''))))
files_pvt = pd.merge(files_pvt, extract_records, on='Library code', how='left')

In [174]:
extract_records

,record_id,sample_id,Library code
0,391,RNA-PV001,PV001
1,392,RNA-PV002,PV002
2,393,RNA-PV003,PV003
3,394,RNA-PV003bgi,PV003bgi
4,395,RNA-PV004bgi,PV004bgi
...,...,...,...
307,698,RNA-PV586,PV586
308,699,RNA-PV587,PV587
309,700,RNA-PV588,PV588
310,701,RNA-PV589,PV589


In [ ]:
files_pvt[files_pvt['record_id'].isna()]

,Library code,Primers,read1,read2,read1_id,read2_id,record_id,sample_id
30,PV111,TAGCTT,PV111_22766_TAGCTT_read1.sub.fastq,PV111_22766_TAGCTT_read2.sub.fastq,63,64,NaN,NaN
229,PV561,ACTTCG,PV561_03636AAB_ACTTCG_read1.sub.fastq,PV561_03636AAB_ACTTCG_read2.sub.fastq,465,466,NaN,NaN
230,PV562,AATGCA,PV562_03637AAB_AATGCA_read1.sub.fastq,PV562_03637AAB_AATGCA_read2.sub.fastq,467,468,NaN,NaN
234,PV566,AACCGG,PV566_03641AAB_AACCGG_read1.sub.fastq,PV566_03641AAB_AACCGG_read2.sub.fastq,475,476,NaN,NaN
236,PV568,CCTAAC,PV568_03643AAB_CCTAAC_read1.sub.fastq,PV568_03643AAB_CCTAAC_read2.sub.fastq,479,480,NaN,NaN
237,PV569,CGTTGG,PV569_03644AAB_CGTTGG_read1.sub.fastq,PV569_03644AAB_CGTTGG_read2.sub.fastq,481,482,NaN,NaN
238,PV570,GTTCAG,PV570_03645AAB_GTTCAG_read1.sub.fastq,PV570_03645AAB_GTTCAG_read2.sub.fastq,483,484,NaN,NaN
239,PV571,CGCAGG,PV571_03646AAB_CGCAGG_read1.sub.fastq,PV571_03646AAB_CGCAGG_read2.sub.fastq,485,486,NaN,NaN
245,PV577,TGCCTA,PV577_03652AAB_TGCCTA_read1.sub.fastq,PV577_03652AAB_TGCCTA_read2.sub.fastq,497,498,NaN,NaN
246,PV578,GCAACG,PV578_03653AAB_GCAACG_read1.sub.fastq,PV578_03653AAB_GCAACG_read2.sub.fastq,499,500,NaN,NaN


In [176]:
files_pvt['Reads code'] = files_pvt['Library code'].apply(lambda x: f'reads-{x}')
files_pvt

,Library code,Primers,read1,read2,read1_id,read2_id,record_id,sample_id,Reads code
0,PV051,ACTTGA,PV051_22518_ACTTGA_read1.sub.fastq,PV051_22518_ACTTGA_read2.sub.fastq,7,8,436,RNA-PV051,reads-PV051
1,PV052,AGTCAA,PV052_22519_AGTCAA_read1.sub.fastq,PV052_22519_AGTCAA_read2.sub.fastq,9,10,437,RNA-PV052,reads-PV052
2,PV053,AGTTCC,PV053_22520_AGTTCC_read1.sub.fastq,PV053_22520_AGTTCC_read2.sub.fastq,11,12,438,RNA-PV053,reads-PV053
3,PV054,ATGTCA,PV054_22521_ATGTCA_read1.sub.fastq,PV054_22521_ATGTCA_read2.sub.fastq,13,14,439,RNA-PV054,reads-PV054
4,PV056,GTGGCC,PV056_22523_GTGGCC_read1.sub.fastq,PV056_22523_GTGGCC_read2.sub.fastq,15,16,441,RNA-PV056,reads-PV056
...,...,...,...,...,...,...,...,...,...
283,PV095,GTGAAA,PV95_22331_GTGAAA_read1.sub.fastq,PV95_22331_GTGAAA_read2.sub.fastq,571,572,477,RNA-PV095,reads-PV095
284,PV096,GTGGCC,PV96_22332_GTGGCC_read1.sub.fastq,PV96_22332_GTGGCC_read2.sub.fastq,575,576,478,RNA-PV096,reads-PV096
285,PV097,GTTTCG,PV97_22531_GTTTCG_read1.sub.fastq,PV97_22531_GTTTCG_read2.sub.fastq,577,578,479,RNA-PV097,reads-PV097
286,PV098,CGTACG,PV98_22334_CGTACG_read1.sub.fastq,PV98_22334_CGTACG_read2.sub.fastq,579,580,480,RNA-PV098,reads-PV098


In [230]:
def create_rawreads_record(entry: Dict[str, Any], project_id: str, people_id: str, sample_type_id: str, assay_id: str):
    attributes = {
        "attribute_map": {
            "Reads code": entry["Reads code"],
            "Extract": entry["record_id"],
            "Forward reads file": entry["read1_id"],
            "Backward reads file": entry["read2_id"],
            "Primers": entry['Primers']
        },
            "tags": [
                "raw read"
            ]
    }
    record = create_record(
        attributes_dict=attributes, project_id=project_id, people_id=people_id,
        sample_type_id=sample_type_id, assay_id=assay_id
    )
    return record


In [231]:
# IMPORTANT TO CHECK THESE ONES BEFORE UPLOAD
people_id = '1'
project_id = '2'
assay_id = '4' # Check these ones carefully
sample_type_id = '4'

create_rawreads_record_ = lambda x: create_rawreads_record(
    x,people_id=people_id, project_id=project_id, assay_id=assay_id, sample_type_id=sample_type_id
)
reads_collection = list(map(create_rawreads_record_, files_pvt.to_dict(orient='records')))
reads_collection[24]


{'data': {'type': 'samples',
  'attributes': {'attribute_map': {'Reads code': 'reads-PV106',
    'Extract': '488',
    'Forward reads file': '53',
    'Backward reads file': '54',
    'Primers': 'TGACCA'},
   'tags': ['raw read']},
  'relationships': {'sample_type': {'data': {'id': '4',
     'type': 'sample_types'}},
   'creators': {'data': [{'id': '1', 'type': 'people'}]},
   'projects': {'data': [{'id': '2', 'type': 'projects'}]},
   'assays': {'data': [{'id': '4', 'type': 'assays'}]}}}}

In [235]:
reads_record = add_samples(reads_collection, "Reads code")

sample # reads-PV051, record 999, added
sample # reads-PV052, record 1000, added
sample # reads-PV053, record 1001, added
sample # reads-PV054, record 1002, added
sample # reads-PV056, record 1003, added
sample # reads-PV057, record 1004, added
sample # reads-PV058, record 1005, added
sample # reads-PV058, record 1006, added
sample # reads-PV062, record 1007, added
sample # reads-PV063, record 1008, added
sample # reads-PV065, record 1009, added
sample # reads-PV066, record 1010, added
sample # reads-PV068, record 1011, added
sample # reads-PV069, record 1012, added
sample # reads-PV072, record 1013, added
sample # reads-PV074, record 1014, added
sample # reads-PV076, record 1015, added
sample # reads-PV001, record 1016, added
sample # reads-PV100, record 1017, added
sample # reads-PV101, record 1018, added
sample # reads-PV102, record 1019, added
sample # reads-PV103, record 1020, added
sample # reads-PV104, record 1021, added
sample # reads-PV105, record 1022, added
sample # reads-PV

In [16]:
def remove_all_samples_from_type(sample_type_id:int, verbose:bool=True):
    try:
        samples_ids = requests.get(f"{api_url}/sample_types/{sample_type_id}", headers=headers).json()['data']['relationships']['samples']['data']
    except KeyError:
        raise IOError(f"Is {sample_type_id} defined??")
    u = []
    for sample in samples_ids:
        for _ in range(5):
            s = requests.delete(f"{api_url}/samples/{sample['id']}", headers=headers)
            time.sleep(0.5)
            if s.status_code == 200:
                break
            elif s.status_code == 500:
                continue
            else:
                break
        if s.status_code == 200 and verbose:
            print("sample with id {0}, removed ".format(sample['id']))
        elif s.status_code == 500 and verbose:
            print("sample with id {0} could not be removed, code{1} ".format(sample['id'], s.status_code))
        

In [17]:
remove_all_samples_from_type(1)

sample with id 2, removed 
sample with id 3, removed 
sample with id 4, removed 
sample with id 5, removed 
sample with id 6, removed 
sample with id 7, removed 
sample with id 8, removed 
sample with id 9, removed 
sample with id 10, removed 
sample with id 11, removed 
sample with id 12, removed 
sample with id 13, removed 
sample with id 14, removed 
sample with id 15, removed 
sample with id 16, removed 
sample with id 17, removed 
sample with id 18, removed 
sample with id 19, removed 
sample with id 20, removed 
sample with id 21, removed 
sample with id 22, removed 
sample with id 23, removed 
sample with id 24, removed 
sample with id 25, removed 
sample with id 26, removed 
sample with id 27, removed 
sample with id 28, removed 
sample with id 29, removed 
sample with id 31, removed 
sample with id 32, removed 
sample with id 33, removed 
sample with id 34, removed 
sample with id 35, removed 
sample with id 36, removed 
sample with id 37, removed 
sample with id 38, removed 
